In [ ]:
import IPython.display
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis

In [ ]:
velocity = na.linspace(-500, 500, axis="wavelength", num=21) * u.km / u.s

In [ ]:
wavelength_rest = 171 * u.AA

In [ ]:
AA = dict(unit=u.AA, equivalencies=u.doppler_optical(wavelength_rest))

In [ ]:
wavelength = velocity.to(**AA)

In [ ]:
position_scene = na.Cartesian2dVectorLinearSpace(
    start=-10 * u.arcsec,
    stop=10 * u.arcsec,
    axis=na.Cartesian2dVectorArray("scene_x", "scene_y"),
    num=na.Cartesian2dVectorArray(64, 64),
)

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 64, axis="sensor_x") * u.pix,
    y=na.arange(0, 64, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_scene = na.SpectralPositionalVectorArray(velocity, position_scene)
coordinates_sensor = na.SpectralPositionalVectorArray(wavelength, position_sensor)

In [ ]:
scene = ctis.scenes.gaussians(
    inputs=coordinates_scene,
    width=na.SpectralPositionalVectorArray(30 * u.km / u.s, 1 * u.arcsec),
)
scene = scene + .1 * scene.outputs.unit

In [ ]:
coordinates_scene.wavelength = wavelength

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    colorbar = na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"scene $y$ ({ax.get_ylabel()})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax2 = ax.twiny()
    na.plt.stairs(
        velocity,
        scene.outputs.mean(("scene_x", "scene_y")),
        ax=ax,
    )
    na.plt.stairs(
        wavelength,
        scene.outputs.mean((("scene_x", "scene_y"))),
        ax=ax2
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=1 * u.cm ** 2,
    timedelta_exposure=10 * u.s,
    plate_scale=.4 * u.arcsec / u.pix,
    dispersion=((100 * u.km / u.s).to(**AA) - wavelength_rest) / u.pix,
    angle=na.linspace(0, 360, num=4, axis="channel", endpoint=False) * u.deg + 45 * u.deg,
    wavelength_ref=wavelength_rest,
    position_ref=32 * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    axis_channel="channel",
    axis_wavelength="wavelength",
    axis_scene_xy=("scene_x", "scene_y"),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
image = instrument.image(scene.outputs, integrate=False)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    label = "dispersion angle = " + instrument.angle.to_string_array("%03d")
    ani, colorbar = na.plt.rgbmovie(
        label,
        image.inputs.wavelength,
        image.inputs.position.x,
        image.inputs.position.y,
        C=image.outputs,
        axis_time="channel",
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=image.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"sensor $x$ ({image.inputs.position.x.unit})")
    ax.set_ylabel(f"sensor $y$ ({image.inputs.position.y.unit})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
images = instrument.image(scene.outputs)

In [ ]:
mart = ctis.inverters.MartInverter(
    instrument=instrument,
    intermediate=True,
)

In [ ]:
inversion = mart(
    images=images.outputs,
    guess=0 * scene.outputs + 1 * scene.outputs.unit,
)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=3,
        gridspec_kw=dict(width_ratios=[.45, .45, .1]),
        constrained_layout=True,
        figsize=(10, 5),
    )
    ax1, ax2, cax = axs
    ax2.sharex(ax1)
    ax2.sharey(ax1)
    na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax1,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    ani, colorbar = na.plt.rgbmovie(
        na.arange(0, inversion.num_iteration, axis=inversion.axis_intermediate),
        scene.inputs.wavelength,
        scene.inputs.position.x,
        scene.inputs.position.y,
        C=inversion.solution,
        axis_time=inversion.axis_intermediate,
        axis_wavelength="wavelength",
        ax=ax2,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"sensor $x$ ({image.inputs.position.x.unit})")
    ax.set_ylabel(f"sensor $y$ ({image.inputs.position.y.unit})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax2 = ax.twiny()
    na.plt.stairs(
        velocity,
        scene.outputs.mean(("scene_x", "scene_y")),
        ax=ax,
    )
    na.plt.stairs(
        wavelength,
        inversion.solution.mean((("scene_x", "scene_y")))[{inversion.axis_intermediate: ~0}],
        ax=ax2,
        color="tab:orange",
    )
    ax.set_xlabel(f"Doppler velocity ({ax.get_xlabel()})")
    ax2.set_xlabel(f"wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"average radiance ({ax.get_ylabel()})")